## 0. Commun fuctions

In [21]:
import csv
import os
import datetime

def log_experiment_csv(csv_file,version_name, input ,start_time, end_time, score, parameters):
  
    duration_seconds = end_time - start_time
    file_name = csv_file
    
    # Prepare data fields
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    
    # Define the row data
    row = {
        "Timestamp": timestamp,
        "Version": version_name,
        "Input": input,
        "Score": score,
        "Duration_Sec": round(duration_seconds, 2),
        "Parameters": parameters
    }
    
    file_exists = os.path.isfile(file_name)
    
    # Write to CSV
    with open(file_name, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        
        # If file is new, write the header first
        if not file_exists:
            writer.writeheader()
            
        writer.writerow(row)

## 1. Greedy function to generate a solution to the problem

In [ ]:
import random

def solve_pizza_problem(pizzas, t2, t3, t4,sort_set=True):
    # Sort pizzas by number of ingredients (descending) to start with 'richer' options
    pizzas.sort(key=lambda x: len(x['ingredients']), reverse=True)
    teams = [(4, t4), (3, t3), (2, t2)]
    if not sort_set:
        random.shuffle(pizzas)
        random.shuffle(teams)
    
    deliveries = []
    used_pizzas = [False] * len(pizzas)
    
    # Process teams from largest to smallest (4 -> 3 -> 2) 
    # because squares of larger numbers yield higher scores.
    for team_size, team_count in teams:
        for _ in range(team_count):
            current_team_pizzas = []
            current_ingredients = set()
            
            # Fill the team requirements
            for _ in range(team_size):
                best_pizza_idx = -1
                max_new_ingredients = -1
                
                # Look for the pizza that adds the most value
                for i in range(len(pizzas)):
                    if not used_pizzas[i]:
                        # Calculate how many NEW ingredients this pizza adds
                        new_count = len(set(pizzas[i]['ingredients']) - current_ingredients)
                        
                        if new_count > max_new_ingredients:
                            max_new_ingredients = new_count
                            best_pizza_idx = i
                        
                        # Optimization: if a pizza adds all its ingredients as new, 
                        # it's a strong candidate; we can break early in large datasets.
                
                if best_pizza_idx != -1:
                    used_pizzas[best_pizza_idx] = True
                    current_team_pizzas.append(pizzas[best_pizza_idx]['id'])
                    current_ingredients.update(pizzas[best_pizza_idx]['ingredients'])
                else:
                    # Not enough pizzas left to fill this team
                    break
            
            if len(current_team_pizzas) == team_size:
                deliveries.append((team_size, current_team_pizzas))
            else:
                # Backtrack: if team wasn't filled, mark pizzas as available again
                for p_id in current_team_pizzas:
                    used_pizzas[p_id] = False
                    
    return deliveries

def score(pizzas,deliveries):
    score_list=[]
    for i in range(len(deliveries)):
        delivered_pizzas=deliveries[i][1]
        ingredients=[]
        for id in delivered_pizzas:
            ingredient_list=pizzas[id]['ingredients']
            for ingredient in ingredient_list:
                ingredients.append(ingredient)
        unique_ingredients=set(ingredients)
        score_list.append(len(unique_ingredients)**2)
        #print(score_list)
    if len(deliveries)==1:
        total_score=score_list[0]
    else:
        total_score=sum(score_list)
    return total_score

## 2. GA Algorithm

In [ ]:
import random
import collections
import numpy as np
import time
import matplotlib.pyplot as plt


# --- Pre-processing ---
# Map ingredients to bitsets for O(1) union operations
def get_pizza_bitsets(pizzas):
    all_ingredients = sorted(list(set(ing for p in pizzas for ing in p['ingredients'])))
    ing_map = {ing: i for i, ing in enumerate(all_ingredients)}
    
    pizza_bitsets = []
    for p in pizzas:
        bits = 0
        for ing in p['ingredients']:
            bits |= (1 << ing_map[ing])
        pizza_bitsets.append(bits)
    return pizza_bitsets

# --- Fitness Function ---
def calculate_fitness(chromosome, pizza_bitsets):
    total_score = 0
    for team_size, pizza_ids in chromosome:
        combined_bits = 0
        for p_id in pizza_ids:
            combined_bits |= pizza_bitsets[p_id]
        
        # Score = (number of unique ingredients)^2 [cite: 106, 109]
        unique_count = bin(combined_bits).count('1')
        total_score += unique_count**2
    return total_score

# --- Crossover ---
def crossover(parent1, parent2, pizza_count):
    # Take half of deliveries from Parent 1
    child = parent1[:len(parent1)//2]
    used_pizzas = {p_id for _, p_list in child for p_id in p_list}
    
    # Fill remaining from Parent 2 if pizzas are available
    for team_size, p_list in parent2:
        if all(p_id not in used_pizzas for p_id in p_list):
            child.append((team_size, p_list))
            used_pizzas.update(p_list)
    return child

# --- Mutation ---
def mutate(chromosome, pizza_bitsets, available_pizzas):
    if len(chromosome) < 2: return chromosome
    
    # 1. Swap Mutation: Swap one pizza between two deliveries of same size
    idx1, idx2 = random.sample(range(len(chromosome)), 2)
    if chromosome[idx1][0] == chromosome[idx2][0]:
        p_list1, p_list2 = list(chromosome[idx1][1]), list(chromosome[idx2][1])
        i1, i2 = random.randint(0, len(p_list1)-1), random.randint(0, len(p_list2)-1)
        
        p_list1[i1], p_list2[i2] = p_list2[i2], p_list1[i1]
        chromosome[idx1] = (chromosome[idx1][0], p_list1)
        chromosome[idx2] = (chromosome[idx2][0], p_list2)
        
    return chromosome


def run_genetic_pizza(pizzas, t2, t3, t4, generations=100, pop_size=20,sort_set=False):
    # Start the total execution timer
    start_total = time.time()
    
    pizza_bits = get_pizza_bitsets(pizzas)
    history = [] 
    
    print(f"--- Starting GA for {generations} generations ---")
    
    # 1. Initialization Time
    start_init = time.time()
    population = []
    for i in range(pop_size):
        shuffled_pizzas = random.sample(pizzas, len(pizzas))
        population.append(solve_pizza_problem(shuffled_pizzas, t2, t3, t4))
    end_init = time.time()
    print(f"Initialization took: {end_init - start_init:.2f} seconds")

    # 2. Evolution Loop
    for gen in range(generations):
        start_gen = time.time()
        
        # Sort by fitness
        population.sort(key=lambda c: calculate_fitness(c, pizza_bits), reverse=True)
        
        current_best_score = calculate_fitness(population[0], pizza_bits)
        history.append(current_best_score)
        
        # Elitism
        new_gen = population[:2]
        
        # Breeding
        while len(new_gen) < pop_size:
            selection_pool = population[:max(2, pop_size // 4)]
            p1, p2 = random.sample(selection_pool, 2)
            child = crossover(p1, p2, len(pizzas))
            child = mutate(child, pizza_bits, len(pizzas))
            new_gen.append(child)
            
        population = new_gen
        end_gen = time.time()
        
        # Print progress and time per generation every 10 generations
        if gen % 10 == 0:
            gen_time = end_gen - start_gen
            print(f"Gen {gen} | Best: {current_best_score} | Time: {gen_time:.4f}s")

    # Final Stats
    end_total = time.time()
    total_duration = end_total - start_total
    
    print("-" * 30)
    print(f"Total Execution Time: {total_duration:.2f} seconds")
    print(f"Average Time per Generation: {total_duration/generations:.4f} seconds")
    print("-" * 30)

    # Plotting the results
    plt.plot(history)
    plt.title(f"GA Progress (Total Time: {total_duration:.2f}s)")
    plt.xlabel("Generation")
    plt.ylabel("Total Score")
    plt.show()
    
    return population[0]

## 3. Tabu Search

In [ ]:
# Tabu Search

import copy

def tabu_search(pizzas, deliveries, iterations, tabu_size):

#def tabu_search(pizzas, deliveries, iterations=3000, tabu_size=100):

    pizza_dict = {p["id"]:p for p in pizzas}

    best_solution = copy.deepcopy(deliveries)
    current_solution = copy.deepcopy(deliveries)

    best_score = score(pizzas,best_solution)

    tabu_list=[]

    all_pizzas=set(pizza_dict.keys())

    for it in range(iterations):

        candidate = copy.deepcopy(current_solution)

        move_type=random.choice(["swap","replace"])

        # ---------------------------------
        # Assuming SWAP pizzas between teams
        # ---------------------------------

        if move_type=="swap" and len(candidate)>=2:

            d1,d2=random.sample(range(len(candidate)),2)

            team1,pizzas1=candidate[d1]
            team2,pizzas2=candidate[d2]

            i=random.randrange(len(pizzas1))
            j=random.randrange(len(pizzas2))

            move=(pizzas1[i],pizzas2[j])

            pizzas1[i],pizzas2[j]=pizzas2[j],pizzas1[i]

            candidate[d1]=(team1,pizzas1)
            candidate[d2]=(team2,pizzas2)

        # --------------------------------------------------------
        # Assuming the possibility if replacing with unused pizza
        # --------------------------------------------------------

        else:

            used=set(p for _,plist in candidate for p in plist)
            unused=list(all_pizzas-used)

            if not unused:
                continue

            d=random.randrange(len(candidate))
            team,pizza_ids=candidate[d]

            idx=random.randrange(len(pizza_ids))

            new_pizza=random.choice(unused)

            move=(pizza_ids[idx],new_pizza)

            pizza_ids[idx]=new_pizza

            candidate[d]=(team,pizza_ids)

        candidate_score=score(pizzas,candidate)

        # Aspiration criterion
        if move in tabu_list and candidate_score<=best_score:
            continue

        current_solution=candidate

        if candidate_score>best_score:

            best_solution=copy.deepcopy(candidate)
            best_score=candidate_score

        tabu_list.append(move)

        if len(tabu_list)>tabu_size:
            tabu_list.pop(0)

        if it % max(1, iterations//10) == 0:
            print("Iteration",it,"Best score:",best_score)

    return best_solution

## 4. Simulated Annealing

In [ ]:
from random import random, randrange, choice
from math import exp

def get_neighbour(deliveries,pizzas):
    new_deliveries = [ (t, p.copy()) for t,p in deliveries ]

    team_idx = randrange(len(new_deliveries))
    team_size, pizza_ids = new_deliveries[team_idx]

    all_used = {p for _,plist in new_deliveries for p in plist}
    unused = [p['id'] for p in pizzas if p['id'] not in all_used]

    if not unused:
        return new_deliveries

    replace_idx = randrange(team_size)
    pizza_ids[replace_idx] = choice(unused)

    new_deliveries[team_idx] = (team_size, pizza_ids)

    return new_deliveries

def simulated_annealing(pizzas, current_solution, current_score,
                        T_max=1000,
                        T_min=0.1,
                        cooling_rate=0.95,
                        iterations_per_temp=5):

    # original parameters: T_max=1000 T_min=0.1 cooling_rate=0.95 iterations_per_temp=100

    start_total = time.time()

    history=[]
    count=0
    T_list=[]

    best_solution = current_solution
    best_score = current_score

    T = T_max

    print(f"Start simulated annealing with maximum temperature= {T_max}; minimum temperature= {T_min}; cooling rate= {cooling_rate}; number of iterations per temperature= {iterations_per_temp} \n")

    while T > T_min:

        start_temp=time.time()

        for _ in range(iterations_per_temp):

            #start_iter=time.time()
            history.append(best_score)
            T_list.append(T)
            neighbour = get_neighbour(current_solution, pizzas)
            neighbour_score = score(pizzas, neighbour)

            delta = neighbour_score - current_score
            

            if delta > 0:
                accept = True
            else:
                accept = random() < exp(delta / T)

            if accept:
                current_solution = neighbour
                current_score = neighbour_score

                if current_score > best_score:
                    best_solution = current_solution
                    best_score = current_score
            #end_iter=time.time()
        count+=1
        T *= cooling_rate
        end_temp=time.time()
        if count % 10: 
            temp_time = end_temp - start_temp
            print(f"Temperature {T} | Best: {best_score} | Time: {temp_time:.4f}s")

    end_total=time.time()
    total_duration = end_total - start_total

    print("-" * 30)
    print(f"Total Execution Time: {total_duration:.2f} seconds")
    print(f"Average Time per Temperature: {total_duration*iterations_per_temp/len(T_list):.4f} seconds")
    print("-" * 30)

    plt.plot(T_list,history)
    plt.title(f"Simulated Annealing Progress (Total Time: {total_duration:.2f}s)")
    plt.xlabel("Temperature")
    plt.ylabel("Total Score")
    plt.show()

    return best_solution, best_score

## 5. Run Experiments